# Week 11: Specialized Topics & Advanced Machine Learning
## Notebook 06: Causal Inference & Propensity Score Matching (PSM) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard Machine Learning models predict **Correlations** ($P(Y \mid X)$):
- *"Users who buy expensive laptops spend more on warranties."*
- If we give every user a laptop, will they buy warranties? **Not necessarily!** Wealth is a **Confounder** causing both.

In observational data where Random Controlled Trials (A/B testing) are unethical or impossible (e.g. smoking vs lung cancer, or credit score vs default):
**Causal Inference** measures the true causal impact of an intervention using **Propensity Score Matching (PSM)**!

---

### 2. Mathematical Foundations: Rubin Causal Model
For individual $i$:
- Treatment indicator: $T_i \in \{0, 1\}$.
- Potential Outcomes: $Y_i(1)$ if treated, $Y_i(0)$ if untreated.
- Fundamental Problem of Causal Inference: We can only ever observe *one* outcome for any individual:
  $$Y_i = T_i Y_i(1) + (1 - T_i) Y_i(0)$$

#### Propensity Score:
Probability of receiving treatment given confounding covariates $X$:
$$e(X_i) = P(T_i = 1 \mid X_i)$$

#### Average Treatment Effect (ATE):
By matching treated units with untreated units having identical propensity scores $e(X)$:
$$\text{ATE} = \mathbb{E}[Y(1) - Y(0)]$$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
N = 1000

# Confounder: User Experience (Years)
experience = np.random.uniform(1, 10, size=N)

# Treatment: Attending Advanced AI Bootcamp (T=1)
# Highly experienced users are more likely to attend (Confounding Selection Bias!)
prob_treatment = 1.0 / (1.0 + np.exp(-(experience - 5.0)))
treatment = np.random.binomial(1, prob_treatment)

# True Causal Effect of Bootcamp on Salary = +$15,000
# But experience ALSO increases salary (+ $8,000 per year)
true_causal_effect = 15000.0
salary = 40000 + (8000 * experience) + (true_causal_effect * treatment) + np.random.normal(0, 3000, N)

df = pd.DataFrame({
    "experience": experience,
    "treatment": treatment,
    "salary": salary
})

# Step 1: Naive Observational Correlation (Biased by Confounder!)
naive_effect = df[df["treatment"] == 1]["salary"].mean() - df[df["treatment"] == 0]["salary"].mean()
print(f"True Causal Effect of Treatment:    USD {true_causal_effect:,.2f}")
print(f"Naive Observational Salary Jump:    USD {naive_effect:,.2f} (Severely Overestimated due to Confounding!)")

# Step 2: Fit Propensity Score Model
ps_model = LogisticRegression()
ps_model.fit(df[["experience"]], df["treatment"])
df["propensity_score"] = ps_model.predict_proba(df[["experience"]])[:, 1]

# Step 3: Implement 1-to-1 Nearest Neighbor Propensity Score Matching
treated = df[df["treatment"] == 1].copy()
control = df[df["treatment"] == 0].copy()

matched_control_salaries = []
for ps in treated["propensity_score"]:
    # Find closest control unit with matching propensity score
    idx = (control["propensity_score"] - ps).abs().idxmin()
    matched_control_salaries.append(control.loc[idx, "salary"])

estimated_ate = treated["salary"].mean() - np.mean(matched_control_salaries)
print(f"Propensity Score Matched ATE:       USD {estimated_ate:,.2f}")
print(f"Estimation Error:                   USD {abs(estimated_ate - true_causal_effect):,.2f}")

### 3. Step-by-Step Code Walkthrough
1. **Selection Bias:** Senior engineers chose to attend the bootcamp, inflating naive salary comparisons to $36,000+ (more than double the real causal effect).
2. **Propensity Score Balancing:** Each bootcamp graduate was paired with a non-attendee who had virtually identical career experience.
3. **Causal Recovery:** PSM estimated the treatment effect at $\sim $14,800, successfully isolating the true $15,000 causal impact.

---

### 4. Industry Applications & Trade-offs
- **Marketing & Retention:** Measuring whether discount coupons actually cause incremental purchases or merely subsidize users who would have purchased anyway.
- **Trade-off:** PSM only controls for **observed** confounders; unobserved confounders (e.g. intrinsic user motivation) can still bias estimates (requires Instrumental Variables or Difference-in-Differences).

---

### 5. Top Interview Questions & Answers
**Q1: What is the "Common Support" assumption in Propensity Score Matching?**
> **Answer:** Common support (or overlap) requires that for every covariate value $X$, both treatment and control have non-zero probabilities: $0 < P(T=1 \mid X) < 1$. If a certain subgroup always receives treatment, no matching control units exist, making causal comparison impossible for that subgroup.

**Q2: What is the difference between ATE (Average Treatment Effect) and ATT (Average Treatment Effect on the Treated)?**
> **Answer:** ATE measures the average causal effect if the entire population were treated vs untreated. ATT measures the causal effect specifically for the subgroup of individuals who actually opted into or received the treatment.